# Film-thickness crop of reconstructed magnetization volume
Loads a `.vti`/`.vtr` reconstruction, crops the z-stack to the true material thickness (symmetric around a known center voxel), and plots the center-y slice (Mz contrast) before/after to verify the crop landed on the right region.

In [ ]:
# --- Cell: imports ---
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import sys
sys.path.append('.')  # folder containing vti_magnetic_import.py
from vti_magnetic_import import load_vti_sample, inspect_vti
import Functions_3D as vis

from scipy.ndimage import gaussian_filter, map_coordinates

import matplotlib.ticker as mticker



## 1. Per-sample film thickness lookup
Same pattern as your existing config -- extended into a small function so it can be called for any sample name.

In [ ]:
def get_sample_cfg(sample):
    """Return {'center1': <z voxel index>, 'film_thickness_nm': <float>} for a sample name."""
    cfg = {}
    if sample in ('665', '665_magconst'):

        cfg['film_thickness_nm'] = 51
    elif sample in ('270', '270_magconst'):

        cfg['film_thickness_nm'] = 66
    elif sample in ('452', '452_magconst', '452_orig'):

        cfg['film_thickness_nm'] = 33
    elif sample in ('105', '105_magconst', '105_orig'):

        cfg['film_thickness_nm'] = 45
    elif sample == 'test_sample_tall':
        # demo-only entry for the synthetic sanity-check volume below --
        # remove once you're pointing this at real data

        cfg['film_thickness_nm'] = 40
    else:
        raise KeyError(f"No crop config defined for sample '{sample}' -- add an entry above.")
    return cfg


## 2. Load the volume
Split into separate config / inspect / load cells so you can edit the path and re-run just that cell, without reloading everything else. The loader auto-detects file format (XML `.vti`/`.vtr`, or legacy `.vtk` in ASCII or binary) from the file content, and auto-detects the vector array name unless you specify one.

In [ ]:
sample_name = '452'

#data_dir = r"C:\Users\emmab\Projects\Vector_Tomography\data_dir"

data_dir = f"C:/Users/emmab/Projects/Vector_Tomography/data_dir/{sample_name}/"
#data_dir = r'C:\Users\emmab\Box\KukrejaLab\Projects\3D Imaging_Emma\Analysis\data_dir'
# data_dir = r"C:\Users\emmab\Projects\Vector_Tomography\data_dir\452_orig"
#test_filename = f"{sample_name}_cropped_magconst_14AUG26_ascii.vtk"
#test_filename = f"{sample_name}_cropped_magconst_22AUG26_ascii.vtk"
#test_filename = f"{sample_name}_cropped_original.vtk"
#test_filename = f"{sample_name}_cropped_normalized.vtk"


####### SIMULATION
#data_dir = r"C:/Users/emmab/Projects/Vector_Tomography/data_dir/452/projections/simulation/"
# test_filename = "452_sim_TV_2px_5z_v2.vtk"

# data_dir = r"C:\Users\emmab\Projects\Vector_Tomography\stripe_dw_2_5z"
# test_filename = "stripes_recon_TV_2_5z_real_angles_200iter_5thick.vti"



data_dir = r"C:\Users\emmab\Projects\Vector_Tomography\data_dir\452"
test_filename = "recon_bsd_TV_5thick.vti"

####### CONST MAG + TV 
# test_filename = f"665_central_film_magconst_TV_dinesh_14OCT26.vtk" ##### THIS IS THE FULL VOLUME CROPPED NEAR THE CENTER !!! 
# test_filename = f"105_constantmag_TV_croppedxyz_22OCT26.vtk" ##### THIS IS THE FULL VOLUME CROPPED NEAR THE CENTER !!! 
#test_filename = f"452_full_cropxy_TV_bsd_NOV26_ascii.vtk"
# # test_filename = f"270_cropped_TV_26OCT26_ascii.vtk"
#######


####### CONST MAG + TV  5 vox thick
# test_filename = f"665_central_film_magconst_TV_dinesh_14OCT26.vtk" ##### THIS IS THE FULL VOLUME CROPPED NEAR THE CENTER !!! 
#test_filename = f"105_recon_TV_5thick.vti" ##### THIS IS THE FULL VOLUME CROPPED NEAR THE CENTER !!! 
#test_filename = f"452_5thin_reproject_TV_real_angles_200iter_15thick.vti"
#test_filename = f"270_recon_TV_9thick.vti"




####### Normalized
# test_filename = f"665_central_film_magconst_TV_dinesh_14OCT26.vtk" ##### THIS IS THE FULL VOLUME CROPPED NEAR THE CENTER !!! 
#test_filename = f"105_recon_TV_5thick.vti" ##### THIS IS THE FULL VOLUME CROPPED NEAR THE CENTER !!! 
#test_filename = f"452_cropped_normalized_ascii_5thick.vtk"
# # test_filename = f"270_cropped_TV_26OCT26_ascii.vtk"



# FSC: CONST MAG + TV 
# test_filename = f"452_half1_fsc_cropxy_TV_bsd_NOV26_ascii.vtk"
#test_filename = f"452_recon_bsd_noreg_fsc_half2_100iter.vti"
# test_filename = f"452_half2_fsc_noreg_bsd_NOV26_ascii.vtk"
#test_filename = f"270_fsc_half2_TV_NOV26_volume.vtk"

#test_filename = f"105_half1_fsc_TV_NOV26_ascii.vtk"


# test_filename = f"452_exactsubset_magconst_19OCT26_ascii.vtk"
#test_filename = f"452_cropped_magconst_19OCT26_ascii.vtk"
#test_filename = f"452_cropped_magconst_bsd_29OCT26_ascii.vtk"



#test_filename = f"452_jeffsnewmagconst_cropped_sqrt2_19OCT26_ascii.vtk"

test_path = os.path.join(data_dir, test_filename)
print(test_path)
inspect_vti(test_path)

In [ ]:
# --- Cell: load the sample and summarize ---
samples = {}
samples[sample_name] = load_vti_sample(test_path, sample_name=sample_name)

s = samples[sample_name]
print("\nLoaded sample summary:")
for k, v in s.items():
    if isinstance(v, np.ndarray):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    else:
        print(f"  {k}: {v}")

print("\nmagnitude min/max:", s["magnitude"].min(), s["magnitude"].max())


In [ ]:
vis.plot_three_slice_fancy(samples[sample_name], field='Mx')

In [ ]:
crop_size = 325
cropped_xy = vis.crop_xy_edges(samples[sample_name], n_pix=crop_size)
print(cropped_xy['shape'])   # (16, 304, 304)
vis.plot_three_slice_fancy(cropped_xy, field='Mz')

# Crop Volume and verify slice
Crops every `(nz, ny, nx)` / `(nz, ny, nx, 3)` array in the sample dict to `film_thickness_nm`, symmetric around len(z-voxel)/2. Returns a new dict -- the original volume is left untouched.

In [ ]:
cfg = get_sample_cfg(sample_name)

nz, ny, nx = cropped_xy['shape']
print(nz, ny, nx)

# # No interp
# samples[sample_name]['cropped'] = vis.crop_volume_by_thickness(
#     samples[sample_name],
#     film_thickness_nm=cfg['film_thickness_nm'], 
#     voxel_size_nm = 8.0/np.sqrt(2),
#     pad_voxels=0
# )
# spacing = (8.0/np.sqrt(2), 8.0/np.sqrt(2), 8.0/np.sqrt(2))
# # spacing = (8.0, 8.0, 8.0)

## interp
# samples[sample_name]['cropped'] = vis.crop_interpolate_volume_by_thickness(
#     samples[sample_name],
#     film_thickness_nm=cfg['film_thickness_nm'],
#     voxel_size_nm = 8.0, #/np.sqrt(2), 
#     target_voxel_size_nm = 2.0
# )
# spacing = (2.0, 2.0, 2.0)


# samples[sample_name]['cropped'] = crop_interpolate_volume_centered(
#     samples[sample_name],
#     film_thickness_nm=cfg['film_thickness_nm'],
#     voxel_size_nm = 8.0, #/np.sqrt(2), 
#     target_voxel_size_nm = 2.0
# )
# spacing = (2.0, 2.0, 2.0)

# samples[sample_name]['cropped'] = crop_interpolate_volume_centered(
#     samples[sample_name],
#     film_thickness_nm=cfg['film_thickness_nm'],
#     voxel_size_nm = 8.0/np.sqrt(2), 
#     target_voxel_size_nm = 8.0/np.sqrt(2)
# )
# spacing = (2.0, 2.0, 2.0)



samples[sample_name]['cropped'] = vis.crop_interpolate_volume_centered_sqrt2_wvox(
    cropped_xy,
    film_thickness_nm=cfg['film_thickness_nm'],
    voxel_size_nm = 8.0, #/np.sqrt(2), 
    target_voxel_size_nm = 8.0, #8.0/np.sqrt(2),
    center_on = 'voxel',
)
# spacing = (8.0/np.sqrt(2), 8.0/np.sqrt(2), 8.0/np.sqrt(2))
#spacing = (2.0, 2.0, 2.0)
spacing = (8.0, 8.0, 8.0)

In [ ]:
nz, ny, nx = cropped_xy['shape']
print(nz, ny, nx)
vis.plot_crop_interpolation_verification_nm(cropped_xy, samples[sample_name]['cropped'], field='Mz')
print(cropped_xy['shape'])
print(spacing)

volume = samples[sample_name]['cropped']

# Visualize Fancy

In [ ]:
vis.plot_three_slice_fancy(volume, field='Mx', voxel_nm = spacing[0])
vis.plot_three_slice_fancy(volume, field='My', voxel_nm = spacing[0])
vis.plot_three_slice_fancy(volume, field='Mz', voxel_nm = spacing[0])
vis.plot_three_slice_fancy(volume, field='Magnitude', voxel_nm = spacing[0])

In [ ]:
# vis.plot_three_slice_fancy(samples[sample_name]['cropped'], field='Mx', voxel_nm = 8.0 / np.sqrt(2))
# vis.plot_three_slice_fancy(samples[sample_name]['cropped'], field='My', voxel_nm = 8.0 / np.sqrt(2))
# vis.plot_three_slice_fancy(samples[sample_name]['cropped'], field='Mz', voxel_nm = 8.0 / np.sqrt(2))
# vis.plot_three_slice_fancy(samples[sample_name]['cropped'], field='Magnitude', voxel_nm = 8.0 / np.sqrt(2))
# plot_three_slice(samples[sample_name]['cropped'], field='My', x_index=33, y_index=33)
# plot_three_slice(samples[sample_name]['cropped'], field='Mz', x_index=75, y_index=75)

In [ ]:
len(volume)

# Crop to make volumes more reasonalbe

In [ ]:
stop

# Save Cropped Data! 

In [ ]:
date_save = 'ground_truth_reproject_99px'


In [ ]:
def save_sample_data(save_dir, sample_name, cropped, spacing=(2.0, 2.0, 2.0), date = None):
    """
    Save the cropped volume and wall-analysis DataFrame for one sample,
    in a format suitable for later cross-sample/material comparison.

    Parameters
    ----------
    save_dir : str
        Directory to save into (created if it doesn't exist).
    sample_name : str
        Identifier for this sample/material, used in filenames and
        stamped into the saved data for later grouping.
    cropped : dict
        The cropped sample dict, expected to contain 'Mx', 'My', 'Mz',
        'magnitude', and 'shape'.
    wall_df : pd.DataFrame
        The per-vertex wall analysis table (normals, psi, chirality, etc.)
    spacing : tuple
        (dz, dy, dx) physical voxel size, in nm.
    z_offset : int
        Offset applied to bring local z-indices back into the original
        volume's index frame (as discussed earlier).
    Delta : float or None
        Domain wall width parameter, if known/relevant, saved as metadata.
    extra_metadata : dict or None
        Any additional key-value info you want stored alongside (e.g.
        material composition, deposition conditions, notes).
    """
    os.makedirs(save_dir, exist_ok=True)

    # --- Save volume arrays ---
    volume_path = os.path.join(save_dir, f'{sample_name}_volume_{date}.npz')
    np.savez_compressed(
        volume_path,
        Mx=cropped['Mx'], My=cropped['My'], Mz=cropped['Mz'],
        magnitude=cropped['magnitude'],
        shape=np.array(cropped['shape']),
        spacing=np.array(spacing),
    )

    print(f"Saved: {volume_path}")
    return volume_path



save_dir = data_dir 

print(save_dir)

save_sample_data(save_dir, sample_name, volume, spacing=spacing,  date = date_save)

In [ ]:
import pyvista as pv


cropped_mesh = volume
print(date_save)

X, Y, Z = cropped_mesh['X'], cropped_mesh['Y'], cropped_mesh['Z']
Mx, My, Mz = cropped_mesh['Mx'], cropped_mesh['My'], cropped_mesh['Mz']
magnitude = cropped_mesh['magnitude']
mask = cropped_mesh['mask']

grid = pv.StructuredGrid(X, Y, Z)

# Vector field: shape (N, 3), flattened in Fortran order to match VTK's point ordering
M_vec = np.stack([Mx.flatten(order="F"),
                   My.flatten(order="F"),
                   Mz.flatten(order="F")], axis=1)
grid.point_data["M"] = M_vec

# Scalars
grid.point_data["Mx"] = Mx.flatten(order="F")
grid.point_data["My"] = My.flatten(order="F")
grid.point_data["Mz"] = Mz.flatten(order="F")
grid.point_data["magnitude"] = magnitude.flatten(order="F")
grid.point_data["mask"] = mask.flatten(order="F").astype(np.uint8)

grid.save(f"C:/Users/emmab/Projects/Vector_Tomography/data_dir/{sample_name}/{sample_name}_{date_save}_volume.vtk")

#grid.save(f"C:/Users/emmab/Projects/Vector_Tomography/stripe_dw_2_5z/Hybrid/{date_save}_volume.vtk")


In [ ]:
stop

In [ ]:
import tifffile


In [ ]:
save_dir

In [ ]:
def save_components_tiff(vol, save_dir, name):
    """Save Mx, My, Mz as separate float32 TIFF stacks, shape (nz, ny, nx)."""
    os.makedirs(save_dir, exist_ok=True)
    paths = []
    for comp in ('Mx', 'My', 'Mz'):
        path = os.path.join(save_dir, f'{name}_{comp}.tiff')
        tifffile.imwrite(path, vol[comp].astype(np.float32))
        paths.append(path)
    return paths


# paths = save_components_tiff(vol_neel, save_dir, f'neel_stripes_wall_{wall_thickness_px}px_t_{nz}z')
paths = save_components_tiff(volume, save_dir, f'{sample_name}_{date_save}_forwardprojection')
# print(paths)